In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import norm, skew, kurtosis
import datetime as dt
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier
from sklearn.svm import SVC, SVR
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.compose import make_column_selector as selector
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.base import TransformerMixin, BaseEstimator, clone, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

In [57]:
data = pd.read_csv("train.csv")
#test = pd.read_csv("test.csv")
store = pd.read_csv("store.csv")

In [58]:
data.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [ ]:
data.drop(columns=['Customers'], inplace=True)

In [59]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [60]:
data_new = pd.merge(data, store, on='Store', how='left')
data_new.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [61]:
data_new['havePI'] = data_new['PromoInterval'].notna().astype(int)
data_new['haveP2SY'] = data_new['Promo2SinceYear'].notna().astype(int)
data_new['haveP2SW'] = data_new['Promo2SinceWeek'].notna().astype(int)
data_new['haveCOSY'] = data_new['CompetitionOpenSinceYear'].notna().astype(int)
data_new['haveCOSM'] = data_new['CompetitionOpenSinceMonth'].notna().astype(int)
data_new['haveCD'] = data_new['CompetitionDistance'].notna().astype(int)

In [62]:
data_new.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,havePI,haveP2SY,haveP2SW,haveCOSY,haveCOSM,haveCD
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,0,NaN,NaN,NaN,0,0,0,1,1,1
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,1,13.0,2010.0,"Jan,Apr,Jul,Oct",1,1,1,1,1,1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,1,14.0,2011.0,"Jan,Apr,Jul,Oct",1,1,1,1,1,1
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,...,0,NaN,NaN,NaN,0,0,0,1,1,1
4,5,5,2015-07-31,4822,559,1,1,0,1,a,...,0,NaN,NaN,NaN,0,0,0,1,1,1


In [63]:
data_new['PromoInterval'].value_counts().sort_values(ascending=False)

PromoInterval
Jan,Apr,Jul,Oct     293122
Feb,May,Aug,Nov     118596
Mar,Jun,Sept,Dec     97460
Name: count, dtype: int64

In [64]:
data_new.shape

(1017209, 24)

In [65]:
data_new['Store'].value_counts().sort_values(ascending=False)

Store
1       942
2       942
3       942
4       942
5       942
       ... 
1094    758
1102    758
1104    758
1107    758
1109    758
Name: count, Length: 1115, dtype: int64

In [66]:
missing = data_new.isnull().sum()
miss_cols = missing[missing>0]
print(miss_cols)

CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64


In [70]:
impute = SimpleImputer(strategy='constant', fill_value=0)
train = pd.DataFrame(impute.fit_transform(data_new), columns=data_new.columns, index=data_new.index)
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,havePI,haveP2SY,haveP2SW,haveCOSY,haveCOSM,haveCD
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,0,0,0,0,0,0,0,1,1,1
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,1,13.0,2010.0,"Jan,Apr,Jul,Oct",1,1,1,1,1,1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,1,14.0,2011.0,"Jan,Apr,Jul,Oct",1,1,1,1,1,1
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,...,0,0,0,0,0,0,0,1,1,1
4,5,5,2015-07-31,4822,559,1,1,0,1,a,...,0,0,0,0,0,0,0,1,1,1


In [71]:
cat_cols = ['StoreType', 'Assortment', 'PromoInterval']
train[cat_cols] = train[cat_cols].astype(str)
ohe = OneHotEncoder()
col = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
], remainder='passthrough', verbose_feature_names_out=False,)

cated_array = col.fit_transform(train)

cated_train = pd.DataFrame(
    cated_array,
    columns=col.get_feature_names_out(),
    index=train.index,
)

print(cated_train.shape)


(1017209, 32)


In [72]:
cated_train.head()

,StoreType_a,StoreType_b,StoreType_c,StoreType_d,Assortment_a,Assortment_b,Assortment_c,PromoInterval_0,"PromoInterval_Feb,May,Aug,Nov","PromoInterval_Jan,Apr,Jul,Oct",...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,havePI,haveP2SY,haveP2SW,haveCOSY,haveCOSM,haveCD
0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2007.0,1,13.0,2010.0,1,1,1,1,1,1
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2006.0,1,14.0,2011.0,1,1,1,1,1,1
3,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,2009.0,0,0,0,0,0,0,1,1,1
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2015.0,0,0,0,0,0,0,1,1,1


In [73]:
X_train = cated_train.set_index('Date')
X_train.head()

,StoreType_a,StoreType_b,StoreType_c,StoreType_d,Assortment_a,Assortment_b,Assortment_c,PromoInterval_0,"PromoInterval_Feb,May,Aug,Nov","PromoInterval_Jan,Apr,Jul,Oct",...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,havePI,haveP2SY,haveP2SW,haveCOSY,haveCOSM,haveCD
Date,,,,,,,,,,,,,,,,,,,,,
2015-07-31,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1
2015-07-31,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2007.0,1,13.0,2010.0,1,1,1,1,1,1
2015-07-31,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,2006.0,1,14.0,2011.0,1,1,1,1,1,1
2015-07-31,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,2009.0,0,0,0,0,0,0,1,1,1
2015-07-31,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2015.0,0,0,0,0,0,0,1,1,1


In [77]:
df = X_train.sort_values(['Store', X_train.index.name]).reset_index()
df = df.set_index('Date')
df.head()

,StoreType_a,StoreType_b,StoreType_c,StoreType_d,Assortment_a,Assortment_b,Assortment_c,PromoInterval_0,"PromoInterval_Feb,May,Aug,Nov","PromoInterval_Jan,Apr,Jul,Oct",...,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,havePI,haveP2SY,haveP2SW,haveCOSY,haveCOSM,haveCD
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1
2013-01-02,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1
2013-01-03,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1
2013-01-04,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1
2013-01-05,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,2008.0,0,0,0,0,0,0,1,1,1


In [80]:
df.index = pd.to_datetime(df.index)

In [81]:
store_categories = df['Store'].astype('category').cat.categories

In [ ]:
def base_features(df):
    df = df.copy()

    
    for lag in [1, 2, 3, 7, 14, 21, 28, 35, 42]:
        df[f'lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)

    
    for w in [7, 14, 21, 28, 35, 42]:
        df[f'roll_mean_{w}'] = (
            df.groupby('Store')['Sales']
              .transform(lambda s: s.shift(1).rolling(w).mean())
        )
        df[f'roll_std_{w}'] = (
            df.groupby('Store')['Sales']
              .transform(lambda s: s.shift(1).rolling(w).std())
        )


    df['is_weekend']  = (df['DayOfWeek']>6).astype(int)
    df['month']      = df.index.month
    df['weekofyear'] = df.index.isocalendar().week.astype(int)

    df['Store'] = pd.Categorical(df['Store'], categories=store_categories)
    return df

df_feat = base_features(df)

In [86]:
df_feat.shape

(1017209, 46)

In [88]:
df_feat.head(10)

,StoreType_a,StoreType_b,StoreType_c,StoreType_d,Assortment_a,Assortment_b,Assortment_c,PromoInterval_0,"PromoInterval_Feb,May,Aug,Nov","PromoInterval_Jan,Apr,Jul,Oct",...,lag_21,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_21,roll_std_21,is_weekend,month,weekofyear
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1
2013-01-02,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1
2013-01-03,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1
2013-01-04,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1
2013-01-05,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1
2013-01-06,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1
2013-01-07,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2
2013-01-08,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,3788.000000,2752.283961,NaN,NaN,NaN,NaN,0,1,2
2013-01-09,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,NaN,4585.142857,2231.018633,NaN,NaN,NaN,NaN,0,1,2


In [ ]:
train_df = df_feat.drop(columns=['Sales'])
y_train = df_feat['Sales']

In [ ]:
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=5,
    n_jobs=1,
    random_state=42,
)
model.fit(train_df, y_train)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error


df = df.sort_index()

X = df.drop(columns=['sales'])
y = df['sales']


tscv = TimeSeriesSplit(n_splits=5, gap=0)

results = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    pip.fit(X_tr, y_tr)
    train_pred = pip.predict(X_tr)
    test_pred  = pip.predict(X_te)

    results.append({
        'fold': fold,
        'train_n': len(tr_idx),
        'test_n':  len(te_idx),
        'train_start': X_tr.index.min(),
        'train_end':   X_tr.index.max(),
        'test_start':  X_te.index.min(),
        'test_end':    X_te.index.max(),
        'train_mae':   mean_absolute_error(y_tr, train_pred),
        'test_mae':    mean_absolute_error(y_te, test_pred),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print("\nMean test MAE:", results_df['test_mae'].mean())
print("Mean train MAE:", results_df['train_mae'].mean())